In [1]:
%load_ext autoreload
%autoreload 2

from zrp.zrp import ZRP
import pandas as pd
import numpy as np

mapping = {
    "FIRST_NAME": "first_name", 
    "MIDDLE_NAME": "middle_name", 
    "LAST_NAME": "last_name", 
    "HOUSE_NUMBER": "house_number", 
    "STREET_ADDRESS": "street_address", 
    "CITY": "city", 
    "STATE": "state", 
    "ZIP_CODE": "zip_code",
    "RECID": "ZEST_KEY",
}

df = pd.read_parquet("data/temp.parquet")

df["ADDRESS"] = (
    df["ADDRESS"]
    .astype(str)
    .str.replace(r"[^\x00-\x7F]+", "", regex=True)
)

df["address_split"] = df["ADDRESS"].str.split(" ")

df["HOUSE_NUMBER"] = (
    df["address_split"]
    .str[0]
    .str.strip()
)

# keep only numeric house numbers
df["HOUSE_NUMBER"] = (
    pd.to_numeric(df["HOUSE_NUMBER"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df["STREET_ADDRESS"] = np.where(
    df["HOUSE_NUMBER"].isna(),
    df["ADDRESS"],
    df["address_split"].str[1:].str.join(" ")
)

df["ZIP_CODE"] = df["ZIP_CODE"].astype("string")
df["RECID"] = df["RECID"].astype("string")

for c in ("CITY", "STATE", "ZIP_CODE", "HOUSE_NUMBER", "STREET_ADDRESS"):
    df[c] = df[c].replace("", pd.NA)
    
df = df.rename(columns=mapping)

df = df[list(mapping.values())]

mask = ~(
    df["house_number"].isna()
    & df["city"].isna()
    & df["state"].isna()
    & (
        df["street_address"].isna()
        | (df["street_address"] == "CONFIDENTIAL")
    )
)

df = df.loc[mask].reset_index(drop=True)

df.shape

(10000, 9)

In [2]:
df = df[df['street_address'].str.strip().isin(['KNOLLWOOD DR', 'EARL KIMBER RD'])]
df = df[df['ZEST_KEY'].isin(['45916', '57839'])]
df

,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,ZEST_KEY
4768,AKIL,AJAMU,GRAVES,1406,KNOLLWOOD DR,BURLINGTON,NC,27217,45916
6011,TROY,CLAYTON,HUNDLEY,241,EARL KIMBER RD,BURLINGTON,NC,27217,57839


In [3]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
folder = Path('./artifacts')
for file in folder.iterdir():
	file.unlink()

zest_race_predictor = ZRP()
output = zest_race_predictor.transform(df)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Directory already exists
####################################
Processing rows: 0:25000
####################################
Data is loaded
   [Start] Validating input data
     Number of observations: 2
     Is key unique: True
Directory already exists
   [Completed] Validating input data

   Formatting P1
   Formatting P2
   reduce whitespace

[Start] Preparing geo data

  The following states are included in the data: ['NC']


  0%|          | 0/1 [00:00<?, ?it/s]

   ... on state: NC

   Data is loaded
   [Start] Processing geo data
      ...address cleaning


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 10 concurrent workers.
100%|██████████| 2/2 [00:00<00:00, 6345.39it/s]

      ...replicating address
         ...Base
         ...Number processing...
         House number dataframe expansion is complete! (n=2)
         ...Base
         ...Map street suffixes...
         ...Mapped & split by street suffixes...
         ...Number processing...

         Address dataframe expansion is complete! (n=2)
      ...formatting
   [Completed] Processing geo data
   [Start] Mapping geo data


...

100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

      ...mapping
   [Completed] Validating input geo data
Directory already exists
...Output saved
   [Completed] Mapping geo data

[Completed] Preparing geo data

[Start] Preparing ACS data
   [Start] Validating ACS input data
     Number of observations: 2
     Is key unique: True

   [Completed] Validating ACS input data

   ...loading ACS lookup tables


   ... combining ACS & user input data
 ...Copy dataframes
 ...Block group
 ...Census tract
 ...Zip code
 ...No match
 ...Merge
 ...Merging complete
[Complete] Preparing ACS data

   [Start] Validating pipeline input data
     Number of observations: 6
     Is key unique: False
   [Completed] Validating pipeline input data



100%|██████████| 1/1 [00:00<00:00, 3521.67it/s]


   ...Proxies generated
...Output saved
...Output saved


In [3]:
zrp_results = pd.read_parquet("artifacts/Zest_Geocoded__2019__37_1.parquet")
zrp_results

,first_name,middle_name,last_name,house_number,street_address,city,state,zip_code,zest_in_state_fips,ZEST_KEY_COL,FROMHN_LEFT,TOHN_LEFT,GEOID_CT,GEOID_BG,GEOID_ZIP,GEOID
ZEST_KEY,,,,,,,,,,,,,,,,
45916,AKIL,AJAMU,GRAVES,1406,KNOLLWOOD DR,BURLINGTON,NC,27217,37,45916,,,37183920800,371839208001,27217,None
57839,TROY,CLAYTON,HUNDLEY,241,EARL KIMBER RD,BURLINGTON,NC,27217,37,57839,,,37001021400,370010214002,27217,None


In [5]:
# Modify df for cencus API
df["street"] = (
	df["house_number"].fillna("").str.strip()
    + " "
    + df["street_address"].fillna("").str.strip()
).str.strip()

batch_df = df[[
    "ZEST_KEY",
    "street",
    "city",
    "state",
    "zip_code"
]]

batch_df = batch_df.rename(columns={"zip_code": "zip"})

# Trimming df due to batch size limit
batch_df = batch_df.iloc[:9900]

batch_df.to_csv(
    "census_batch.csv",
    index=False,
    header=True
)


In [6]:
import requests

url = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"

files = {
    "addressFile": open("census_batch.csv", "rb")
}

data = {
    "benchmark": "Public_AR_Current",
    "vintage": "Current_Current"
}

response = requests.post(url, files=files, data=data)

with open("census_batch_results.csv", "wb") as f:
    f.write(response.content)

In [7]:
zrp_results.to_csv('py311results.csv')

In [4]:
# reads batch results to a dataframe
cols = [
    "ZEST_KEY",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "coordinates",
    "tiger_line_id",
    "side",
    "state_fips",
    "county_fips",
    "tract",
    "block",
]

df_census = pd.read_csv(
    "census_batch_results.csv",
    header=None,
    names=cols,
    dtype=str
)

df_census["GEOID_CT"] = (
    df_census["state_fips"]
    + df_census["county_fips"]
    + df_census["tract"]
)

df_census["GEOID_BG"] = (
    df_census["GEOID_CT"]
    + df_census["block"].str[0]
)

df_census.columns

Index(['ZEST_KEY', 'input_address', 'match_status', 'match_type',
       'matched_address', 'coordinates', 'tiger_line_id', 'side', 'state_fips',
       'county_fips', 'tract', 'block', 'GEOID_CT', 'GEOID_BG'],
      dtype='object')

In [5]:
merged = zrp_results.merge(
    df_census[["ZEST_KEY", "GEOID_CT", "GEOID_BG", "state_fips", "county_fips", "tract"]],
    on="ZEST_KEY",
    suffixes=("_zrp", "_census")
)

# drop failed entries
merged = merged.dropna(subset=["GEOID_CT_zrp", "GEOID_CT_census"])

merged["CT_match"] = merged["GEOID_CT_zrp"] == merged["GEOID_CT_census"]
print("CT Id match: ")
print(merged["CT_match"].value_counts(normalize=True))

merged["ct_state_match"] = merged["state_fips"] == merged["GEOID_CT_zrp"].str[:2]
merged["ct_county_match"] = merged["county_fips"] == merged["GEOID_CT_zrp"].str[2:5]
merged["ct_tract_match"] = merged["tract"] == merged["GEOID_CT_zrp"].str[5:]

print("States match: ")
print(merged["ct_state_match"].value_counts(normalize=True))
print("Counties match match: ")
print(merged["ct_county_match"].value_counts(normalize=True))
print("Tracts match: ")
print(merged["ct_tract_match"].value_counts(normalize=True))

print(merged.columns)
print(merged["CT_match"].value_counts(normalize=True))

merged.groupby("ct_state_match")["ct_county_match"].value_counts(normalize=True)
merged[['house_number', 'street_address', 'city', 'state', 'zip_code', 'GEOID_CT_census', 'GEOID_CT_zrp', 'ct_state_match', 'ct_county_match', 'ct_tract_match']].head(n=10)


CT Id match: 
CT_match
False    0.5
True     0.5
Name: proportion, dtype: float64
States match: 
ct_state_match
True    1.0
Name: proportion, dtype: float64
Counties match match: 
ct_county_match
False    0.5
True     0.5
Name: proportion, dtype: float64
Tracts match: 
ct_tract_match
False    0.5
True     0.5
Name: proportion, dtype: float64
Index(['ZEST_KEY', 'first_name', 'middle_name', 'last_name', 'house_number',
       'street_address', 'city', 'state', 'zip_code', 'zest_in_state_fips',
       'ZEST_KEY_COL', 'FROMHN_LEFT', 'TOHN_LEFT', 'GEOID_CT_zrp',
       'GEOID_BG_zrp', 'GEOID_ZIP', 'GEOID', 'GEOID_CT_census',
       'GEOID_BG_census', 'state_fips', 'county_fips', 'tract', 'CT_match',
       'ct_state_match', 'ct_county_match', 'ct_tract_match'],
      dtype='object')
CT_match
False    0.5
True     0.5
Name: proportion, dtype: float64


,house_number,street_address,city,state,zip_code,GEOID_CT_census,GEOID_CT_zrp,ct_state_match,ct_county_match,ct_tract_match
0,1406,KNOLLWOOD DR,BURLINGTON,NC,27217,37001020301,37183920800,True,False,False
1,241,EARL KIMBER RD,BURLINGTON,NC,27217,37001021400,37001021400,True,True,True


In [9]:
py37results = pd.read_csv('py37results.csv')
py311results = pd.read_csv('py311results.csv')
py37results.equals(py311results)
py37results.columns

Index(['ZEST_KEY', 'first_name', 'middle_name', 'last_name', 'house_number',
       'street_address', 'city', 'state', 'zip_code', 'zest_in_state_fips',
       'ZEST_KEY_COL', 'FROMHN_LEFT', 'TOHN_LEFT', 'GEOID_CT', 'GEOID_BG',
       'GEOID_ZIP', 'GEOID'],
      dtype='object')